### ASG Airlines – End-to-End Data Engineering Pipeline

## Project Overview

### This notebook implements an end-to-end data engineering pipeline for ASG Airlines.

### The objective is to ingest the raw airline datasets, assess and improve data quality, apply the required transformations and business rules, protect sensitive passenger information, and prepare reliable analytical datasets for Power BI reporting

### 1. This is the first stage of pipeline were loading the rawdata into the notebook

In [2]:
import pandas as pd

# Load raw datasets
flights = pd.read_excel("C:\\Users\\HP\\Downloads\\UseCase - Airlines.xlsx", sheet_name="flights")
bookings = pd.read_excel("C:\\Users\\HP\\Downloads\\UseCase - Airlines.xlsx", sheet_name="bookings")
passengers = pd.read_excel("C:\\Users\\HP\\Downloads\\UseCase - Airlines.xlsx", sheet_name="passengers")
payments = pd.read_excel("C:\\Users\\HP\\Downloads\\UseCase - Airlines.xlsx", sheet_name="payments")

print("Data loaded successfully!")

Data loaded successfully!


In [3]:
# Displaying the size of each dataset

print("Flights:", flights.shape)
print("Bookings:", bookings.shape)
print("Passengers:", passengers.shape)
print("Payments:", payments.shape)

Flights: (1020, 7)
Bookings: (1000, 9)
Passengers: (1039, 9)
Payments: (1000, 4)


In [4]:
# Preview the raw datasets

display(flights.head())
display(bookings.head())
display(passengers.head())
display(payments.head())

,flight_id,airline,source,destination,departure_time,arrival_time,duration
0,SJ010,SpiceJet,CCU,MAA,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,02:54:00
1,AI155,Air India,BOM,CCU,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,01:48:00
2,UK094,Vistara,BOM,CCU,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,01:45:00
3,AI245,Air India,BOM,CCU,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704,02:36:00
4,AI192,Air India,MAA,BOM,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,04:59:00


,booking_id,passenger_id,flight_id,booking_date,status,passport_number,seat_number,emergency_contact_name,emergency_contact_phone
0,B1000,P1591,AI192,2025-06-14 11:37:36.951,CANCELLED,P1945887,3D,Isaac Bakshi,+91-6478475128
1,B1001,P1803,6F026,2025-11-02 11:37:36.951,CANCELLED,L3482012,18A,Anvi Konda,+91-6647078662
2,B1002,P1083,SJ010,2025-08-25 11:37:36.951,CANCELLED,G8507659,30C,Udant Dewan,+91-8405938220
3,B1003,P1364,AI069,2025-12-30 11:37:36.951,CONFIRMED,M0891776,33A,Harsh Chahal,+91-6264636839
4,B1004,P1885,UK003,2025-10-02 11:37:36.951,PENDING,N5742231,25C,Pahal Balay,+91-9336478266


,passenger_id,first_name,last_name,age,gender,email,phone,aadhaar_id,date_of_birth
0,P1000,Vivaan,Chatterjee,52,F,vivaan.chatterjee@gmail.com,+91-6896233790,433218196001,1974-04-08
1,P1001,Krishna,Reddy,15,M,krishna.reddy@hotmail.com,+91-6702632297,386379402654,2011-03-07
2,P1002,Myra,Naidu,72,M,myra.naidu@outlook.com,+91-6199585092,615594078161,1954-09-10
3,P1003,Myra,Mishra,61,F,myra.mishra@hotmail.com,+91-8719927151,310341316475,1965-03-12
4,P1004,Saanvi,Banerjee,21,M,saanvi.banerjee@outlook.com,+91-7819595113,419283276483,2005-11-11


,payment_id,booking_id,amount,payment_method
0,PAY1000,B1116,9883.49,NETBANKING
1,PAY1001,B1738,8457.96,NETBANKING
2,PAY1002,B1873,6495.37,UPI
3,PAY1003,B1914,5079.38,NETBANKING
4,PAY1004,B1967,12518.31,CARD


## 2. Data Understanding

### Before applying any cleaning or transformation, the structure and characteristics of each dataset are examined.

In [5]:
# Dataset overview

datasets = {
    "Flights": flights,
    "Bookings": bookings,
    "Passengers": passengers,
    "Payments": payments
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} columns")

Flights: 1,020 rows × 7 columns
Bookings: 1,000 rows × 9 columns
Passengers: 1,039 rows × 9 columns
Payments: 1,000 rows × 4 columns


In [6]:
# Column names and data types

for name, df in datasets.items():
    print(f"\n{'=' * 50}")
    print(f"{name}")
    print(f"{'=' * 50}")
    print(df.dtypes)


Flights
flight_id                 object
airline                   object
source                    object
destination               object
departure_time    datetime64[ns]
arrival_time      datetime64[ns]
duration                  object
dtype: object

Bookings
booking_id                         object
passenger_id                       object
flight_id                          object
booking_date               datetime64[ns]
status                             object
passport_number                    object
seat_number                        object
emergency_contact_name             object
emergency_contact_phone            object
dtype: object

Passengers
passenger_id             object
first_name               object
last_name                object
age                       int64
gender                   object
email                    object
phone                    object
aadhaar_id                int64
date_of_birth    datetime64[ns]
dtype: object

Payments
payment_id        ob

In [7]:
# Schema summary

for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * 60)

    schema = pd.DataFrame({
        "Column": df.columns.tolist(),
        "Data Type": df.dtypes.astype(str).tolist(),
        "Non-Null Count": df.notna().sum().tolist(),
        "Unique Values": df.nunique(dropna=True).tolist()
    })

    display(schema)


Flights
------------------------------------------------------------


,Column,Data Type,Non-Null Count,Unique Values
0,flight_id,object,1020,1004
1,airline,object,979,5
2,source,object,1020,6
3,destination,object,1020,6
4,departure_time,datetime64[ns],1020,981
5,arrival_time,datetime64[ns],1020,969
6,duration,object,1020,270



Bookings
------------------------------------------------------------


,Column,Data Type,Non-Null Count,Unique Values
0,booking_id,object,1000,1000
1,passenger_id,object,1000,636
2,flight_id,object,1000,984
3,booking_date,datetime64[ns],1000,552
4,status,object,955,4
5,passport_number,object,1000,1000
6,seat_number,object,1000,202
7,emergency_contact_name,object,1000,998
8,emergency_contact_phone,object,1000,1000



Passengers
------------------------------------------------------------


,Column,Data Type,Non-Null Count,Unique Values
0,passenger_id,object,1039,1000
1,first_name,object,1039,31
2,last_name,object,1029,38
3,age,int64,1039,89
4,gender,object,1039,2
5,email,object,1039,1039
6,phone,object,1039,1039
7,aadhaar_id,int64,1039,1039
8,date_of_birth,datetime64[ns],1039,1011



Payments
------------------------------------------------------------


,Column,Data Type,Non-Null Count,Unique Values
0,payment_id,object,1000,1000
1,booking_id,object,1000,637
2,amount,object,952,922
3,payment_method,object,1000,3


### Data relations i have considering the star schema why because here we can easily observe that booking table only one which connected with all tables booking table is the centre of attraction and other tables dimensions of that.

## 3. Data Qulaity Check

In [8]:
# Missing value

for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * 50)

    missing = df.isna().sum()
    missing_pct = (missing / len(df) * 100).round(2)

    quality = pd.DataFrame({
        "Missing Count": missing
    })

    display(quality[quality["Missing Count"] > 0])


Flights
--------------------------------------------------


,Missing Count
airline,41



Bookings
--------------------------------------------------


,Missing Count
status,45



Passengers
--------------------------------------------------


,Missing Count
last_name,10



Payments
--------------------------------------------------


,Missing Count
amount,48


In [9]:
# Exact duplicate

for name, df in datasets.items():
    duplicate_count = df.duplicated().sum()

    print(f"{name}: {duplicate_count} exact duplicate rows")

Flights: 15 exact duplicate rows
Bookings: 0 exact duplicate rows
Passengers: 0 exact duplicate rows
Payments: 0 exact duplicate rows


In [10]:
# Check current data types

for name, df in datasets.items():
    print(f"\n{name}")
    print(df.dtypes)


Flights
flight_id                 object
airline                   object
source                    object
destination               object
departure_time    datetime64[ns]
arrival_time      datetime64[ns]
duration                  object
dtype: object

Bookings
booking_id                         object
passenger_id                       object
flight_id                          object
booking_date               datetime64[ns]
status                             object
passport_number                    object
seat_number                        object
emergency_contact_name             object
emergency_contact_phone            object
dtype: object

Passengers
passenger_id             object
first_name               object
last_name                object
age                       int64
gender                   object
email                    object
phone                    object
aadhaar_id                int64
date_of_birth    datetime64[ns]
dtype: object

Payments
payment_id        ob

### Flight Identifier Quality
### In the instruction i have that they mention some of the id may contain corrupted we have find that cullprits

In [11]:
import re

flight_id_pattern = r"^[A-Z0-9]{2}\d{3}$"

invalid_flight_ids = flights[
    ~flights["flight_id"].astype(str).str.match(flight_id_pattern, na=False)
]

print("Invalid flight IDs:", len(invalid_flight_ids))

display(invalid_flight_ids[["flight_id"]].drop_duplicates())

Invalid flight IDs: 0


,flight_id


### Since I have done the quality check but I haven't find any corrupted id's because the pattern for all the id's is same  

In [12]:
# Identify flights where arrival time occurs before departure time

arrival_before_departure = flights[
    flights["arrival_time"] < flights["departure_time"]
]

print("Arrival before departure:", len(arrival_before_departure))

display(
    arrival_before_departure[
        ["flight_id", "source", "destination",
         "departure_time", "arrival_time"]
    ].head(10)
)

Arrival before departure: 1


,flight_id,source,destination,departure_time,arrival_time
355,SJ192,HYD,BOM,2026-04-19 18:45:42,2026-04-18 23:45:42


#### Time Quality Finding

#### One flight record (`SJ192`) contains an inconsistent timestamp relationship:

#### - Departure: 19-Apr-2026 18:45:42
#### - Arrival: 18-Apr-2026 23:45:42

#### The arrival timestamp is one calendar day before the departure timestamp. This is different from a standard overnight flight, where the arrival time is earlier but occurs on the following calendar day.

#### The record is therefore flagged as a temporal data-quality issue and will be handled separately during the transformation stage rather than being automatically treated as an overnight flight.

In [13]:
quality_summary = []

for name, df in datasets.items():
    quality_summary.append({
        "Dataset": name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Missing Cells": int(df.isna().sum().sum()),
        "Exact Duplicates": int(df.duplicated().sum())
    })

quality_summary = pd.DataFrame(quality_summary)

display(quality_summary)

,Dataset,Rows,Columns,Missing Cells,Exact Duplicates
0,Flights,1020,7,41,15
1,Bookings,1000,9,45,0
2,Passengers,1039,9,10,0
3,Payments,1000,4,48,0


## Referential check between the tables

In [14]:
# Referential integrity checks

missing_flights = bookings[
    ~bookings["flight_id"].isin(flights["flight_id"])
]

missing_passengers = bookings[
    ~bookings["passenger_id"].isin(passengers["passenger_id"])
]

missing_bookings = payments[
    ~payments["booking_id"].isin(bookings["booking_id"])
]

print("True")

True


#### Data Quality Assessment – Conclusion

#### The raw datasets were assessed for missing values, duplicate records, schema and datatype consistency, identifier validity, temporal inconsistencies, and referential integrity.

#### The assessment identified the issues that require treatment during the cleaning and transformation stages.

#### Importantly, records are not removed solely because they appear unusual. Each issue will be handled according to its data meaning and applicable business rule.

#### The identified data quality findings will now be addressed systematically in the subsequent pipeline stages.

### 4. Duplicate Handling

In [15]:
# Count exact duplicates before cleaning

print("Exact duplicates before cleaning:")

for name, df in datasets.items():
    print(f"{name}: {df.duplicated().sum()}")

Exact duplicates before cleaning:
Flights: 15
Bookings: 0
Passengers: 0
Payments: 0


In [16]:
# Remove exact duplicate rows

flights = flights.drop_duplicates().copy()
bookings = bookings.drop_duplicates().copy()
passengers = passengers.drop_duplicates().copy()
payments = payments.drop_duplicates().copy()

In [17]:
# Verify exact duplicates after cleaning

print("Exact duplicates after cleaning:")

for name, df in {
    "Flights": flights,
    "Bookings": bookings,
    "Passengers": passengers,
    "Payments": payments
}.items():
    print(f"{name}: {df.duplicated().sum()}")

Exact duplicates after cleaning:
Flights: 0
Bookings: 0
Passengers: 0
Payments: 0


### 5. Missing value imputation

In [18]:
# Create a mapping from flight prefix to the observed airline name

valid_airlines = ["IndiGo", "Air India", "SpiceJet", "Vistara"]

airline_mapping = (
    flights[flights["airline"].isin(valid_airlines)]
    .assign(
        airline_prefix=lambda df: df["flight_id"].str[:2]
    )
    .drop_duplicates("airline_prefix")
    .set_index("airline_prefix")["airline"]
)

display(airline_mapping)

airline_prefix
SJ     SpiceJet
AI    Air India
UK      Vistara
6F       IndiGo
Name: airline, dtype: object

In [19]:
# Derive airline prefix from flight_id
airline_prefix = flights["flight_id"].str[:2]

# Fill missing and UNKNOWN airline values using the validated mapping
mask = flights["airline"].isna() | (flights["airline"] == "UNKNOWN")

flights.loc[mask, "airline"] = airline_prefix[mask].map(airline_mapping)

print("Missing airline values:", flights["airline"].isna().sum())
print("UNKNOWN airline values:", (flights["airline"] == "UNKNOWN").sum())

Missing airline values: 0
UNKNOWN airline values: 0


#### here i have imputed the missing values in the airline prefix 2 characters of airline id why because by observing the pattern of existing rows by manually i came to these decision

### 6. Data Standardization

In [20]:
# Inspect distinct categorical values before standardization

for column in ["airline", "source", "destination"]:
    print(f"\n{column}")
    print("-" * 40)
    print(sorted(flights[column].dropna().unique()))


airline
----------------------------------------
['Air India', 'IndiGo', 'SpiceJet', 'Vistara']

source
----------------------------------------
['BLR', 'BOM', 'CCU', 'DEL', 'HYD', 'MAA']

destination
----------------------------------------
['BLR', 'BOM', 'CCU', 'DEL', 'HYD', 'MAA']


In [21]:
# Validate standardized categorical fields

print("Missing airline:", flights["airline"].isna().sum())
print("UNKNOWN airline:", (flights["airline"] == "UNKNOWN").sum())

print(
    "Non-uppercase source values:",
    (flights["source"] != flights["source"].str.upper()).sum()
)

print(
    "Non-uppercase destination values:",
    (flights["destination"] != flights["destination"].str.upper()).sum()
)

print(
    "Airline values with leading/trailing spaces:",
    (flights["airline"] != flights["airline"].str.strip()).sum()
)

print(
    "Source values with leading/trailing spaces:",
    (flights["source"] != flights["source"].str.strip()).sum()
)

print(
    "Destination values with leading/trailing spaces:",
    (flights["destination"] != flights["destination"].str.strip()).sum()
)

Missing airline: 0
UNKNOWN airline: 0
Non-uppercase source values: 0
Non-uppercase destination values: 0
Airline values with leading/trailing spaces: 0
Source values with leading/trailing spaces: 0
Destination values with leading/trailing spaces: 0


## 7. Data Transformation

In [22]:
# Convert flight timestamps to datetime

flights["departure_time"] = pd.to_datetime(
    flights["departure_time"]
)

flights["arrival_time"] = pd.to_datetime(
    flights["arrival_time"]
)

print("Departure dtype:", flights["departure_time"].dtype)
print("Arrival dtype:", flights["arrival_time"].dtype)

Departure dtype: datetime64[ns]
Arrival dtype: datetime64[ns]


In [23]:
# Find the  potential overnight flights based on clock time where acutal earlier than the depature time

departure_clock = flights["departure_time"].dt.time
arrival_clock = flights["arrival_time"].dt.time

overnight_candidates = flights[
    flights["arrival_time"].dt.time < flights["departure_time"].dt.time
]

print("Potential overnight flights:", len(overnight_candidates))

display(
    overnight_candidates[
        ["flight_id", "departure_time", "arrival_time"]
    ].head(10)
)

Potential overnight flights: 122


,flight_id,departure_time,arrival_time
0,SJ010,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701
1,AI155,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703
2,UK094,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702
3,AI245,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704
4,AI192,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703
5,SJ158,2026-04-20 23:05:41.703,2026-04-21 01:24:41.703
6,6F196,2026-04-20 23:04:41.703,2026-04-21 00:47:41.703
7,AI080,2026-04-20 23:03:41.702,2026-04-21 00:35:41.702
10,AI137,2026-04-20 22:49:41.702,2026-04-21 01:17:41.702
11,6F026,2026-04-20 22:46:42.000,2026-04-21 03:13:42.000


In [24]:
# Identify records where arrival date is earlier than departure date 

invalid_date_order = flights[
    flights["arrival_time"].dt.date < flights["departure_time"].dt.date
]

print("Arrival date earlier than departure date:", len(invalid_date_order))

display(
    invalid_date_order[
        ["flight_id", "departure_time", "arrival_time"]
    ]
)

Arrival date earlier than departure date: 1


,flight_id,departure_time,arrival_time
355,SJ192,2026-04-19 18:45:42,2026-04-18 23:45:42


In [25]:
# Identify valid overnight flights
valid_overnight = (
    (flights["arrival_time"].dt.time < flights["departure_time"].dt.time)
    & (flights["arrival_time"].dt.date >= flights["departure_time"].dt.date)
)

print("Valid overnight flights:", valid_overnight.sum())

Valid overnight flights: 122


In [26]:
# Calculate flight duration for over night fghts we have add 24 hours plus
flights["duration"] = (
    flights["arrival_time"] - flights["departure_time"]
)

flights["duration_hours"] = (
    flights["duration"].dt.total_seconds() / 3600
)

In [27]:
#For finding the anomoly
display(
    flights.loc[
        flights["flight_id"] == "SJ192",
        [
            "flight_id",
            "source",
            "destination",
            "departure_time",
            "arrival_time",
            "duration",
            "duration_hours"
        ]
    ]
)

,flight_id,source,destination,departure_time,arrival_time,duration,duration_hours
355,SJ192,HYD,BOM,2026-04-19 18:45:42,2026-04-18 23:45:42,-1 days +05:00:00,-19.0


In [28]:
# Remove records with invalid flight duration
flights = flights[flights["duration_hours"] >= 0].copy()

print("Flights after removing invalid duration records:", len(flights))

Flights after removing invalid duration records: 1004


In [29]:
print("Invalid duration records remaining:",
      (flights["duration_hours"] < 0).sum())

Invalid duration records remaining: 0


## 8. PII Protection

The passenger and booking datasets contain personally identifiable information (PII) and sensitive information.

The following fields require protection before the data is used for analytical reporting:

### Passengers
- `first_name`
- `last_name`
- `email`
- `phone`
- `aadhaar_id`
- `date_of_birth`

### Bookings
- `passport_number`
- `emergency_contact_name`
- `emergency_contact_phone`

The protection strategy is based on the analytical requirement of each field:

- Aadhaar and passport numbers are hashed because their original values are not required for reporting.
- Email and phone numbers are masked to retain limited information while protecting the full identifier.
- Emergency contact information is excluded from the analytical dataset because it is not required for the requested KPIs.
- Date of birth will be transformed into a less precise analytical representation rather than exposing the complete date.

In [30]:
# Create copies for analytical use
passengers_analytics = passengers.copy()
bookings_analytics = bookings.copy()

print("Passenger analytical dataset:", passengers_analytics.shape)
print("Booking analytical dataset:", bookings_analytics.shape)

Passenger analytical dataset: (1039, 9)
Booking analytical dataset: (1000, 9)


#### Mask and hash PII

#### I use:

#### SHA-256 hashing for aadhaar_id and passport_number
#### Partial masking for email and phone

In [31]:
import hashlib

def hash_value(value):
    if pd.isna(value):
        return value
    return hashlib.sha256(str(value).encode()).hexdigest()


# Hash highly sensitive identifiers
passengers_analytics["aadhaar_id"] = (
    passengers_analytics["aadhaar_id"].apply(hash_value)
)

bookings_analytics["passport_number"] = (
    bookings_analytics["passport_number"].apply(hash_value)
)


# Mask email addresses
def mask_email(email):
    if pd.isna(email):
        return email

    email = str(email)
    username, domain = email.split("@", 1)

    masked_username = username[0] + "***"

    return masked_username + "@" + domain


passengers_analytics["email"] = (
    passengers_analytics["email"].apply(mask_email)
)


# Mask phone numbers
def mask_phone(phone):
    if pd.isna(phone):
        return phone

    phone = str(phone)
    return "*" * max(0, len(phone) - 4) + phone[-4:]


passengers_analytics["phone"] = (
    passengers_analytics["phone"].apply(mask_phone)
)


# Keep only the year from date of birth
passengers_analytics["date_of_birth"] = pd.to_datetime(
    passengers_analytics["date_of_birth"],
    errors="coerce"
).dt.year


# Remove emergency contact PII
bookings_analytics = bookings_analytics.drop(
    columns=["emergency_contact_name", "emergency_contact_phone"]
)

In [32]:
display(passengers_analytics.head())
display(bookings_analytics.head())

,passenger_id,first_name,last_name,age,gender,email,phone,aadhaar_id,date_of_birth
0,P1000,Vivaan,Chatterjee,52,F,v***@gmail.com,**********3790,99466baa9b8fe69a6f31db34c87d50f0e9a488eda788f3...,1974
1,P1001,Krishna,Reddy,15,M,k***@hotmail.com,**********2297,6d05ccbeb5011dd59948b882fff86e1a6f7fc0cf41f659...,2011
2,P1002,Myra,Naidu,72,M,m***@outlook.com,**********5092,3e24ac79e2c956797c22051676644d26204cc4de0f203e...,1954
3,P1003,Myra,Mishra,61,F,m***@hotmail.com,**********7151,4e880723014ee24b39f0213150496f3efb9faa3f62d335...,1965
4,P1004,Saanvi,Banerjee,21,M,s***@outlook.com,**********5113,3ca09c31f18da71ad2bc39cb3cbb387c38a9cd5ad37239...,2005


,booking_id,passenger_id,flight_id,booking_date,status,passport_number,seat_number
0,B1000,P1591,AI192,2025-06-14 11:37:36.951,CANCELLED,59e7e7738c8711b2f49be2c355b25b9bd41b780392aa21...,3D
1,B1001,P1803,6F026,2025-11-02 11:37:36.951,CANCELLED,6a95f336b866dc4d580b61301c1a96016a5ec24e056b40...,18A
2,B1002,P1083,SJ010,2025-08-25 11:37:36.951,CANCELLED,8d413701d70b517478f82789245fa033726307ec10b14c...,30C
3,B1003,P1364,AI069,2025-12-30 11:37:36.951,CONFIRMED,19c4a665e8bc1266b54aee9a9925f0d4179044fbc7be19...,33A
4,B1004,P1885,UK003,2025-10-02 11:37:36.951,PENDING,d9157c803314e083e13c0307a6e506c2beaa8c6e96140c...,25C


In [33]:
print("Aadhaar sample:")
display(passengers_analytics["aadhaar_id"].head())

print("\nPassport sample:")
display(bookings_analytics["passport_number"].head())

print("\nEmail sample:")
display(passengers_analytics["email"].head())

print("\nPhone sample:")
display(passengers_analytics["phone"].head())

print("\nBooking columns:")
print(bookings_analytics.columns.tolist())

Aadhaar sample:


0    99466baa9b8fe69a6f31db34c87d50f0e9a488eda788f3...
1    6d05ccbeb5011dd59948b882fff86e1a6f7fc0cf41f659...
2    3e24ac79e2c956797c22051676644d26204cc4de0f203e...
3    4e880723014ee24b39f0213150496f3efb9faa3f62d335...
4    3ca09c31f18da71ad2bc39cb3cbb387c38a9cd5ad37239...
Name: aadhaar_id, dtype: object


Passport sample:


0    59e7e7738c8711b2f49be2c355b25b9bd41b780392aa21...
1    6a95f336b866dc4d580b61301c1a96016a5ec24e056b40...
2    8d413701d70b517478f82789245fa033726307ec10b14c...
3    19c4a665e8bc1266b54aee9a9925f0d4179044fbc7be19...
4    d9157c803314e083e13c0307a6e506c2beaa8c6e96140c...
Name: passport_number, dtype: object


Email sample:


0      v***@gmail.com
1    k***@hotmail.com
2    m***@outlook.com
3    m***@hotmail.com
4    s***@outlook.com
Name: email, dtype: object


Phone sample:


0    **********3790
1    **********2297
2    **********5092
3    **********7151
4    **********5113
Name: phone, dtype: object


Booking columns:
['booking_id', 'passenger_id', 'flight_id', 'booking_date', 'status', 'passport_number', 'seat_number']


### 9.Data validation

In [34]:
# Validate cleaned flight data

print("Exact duplicate rows:", flights.duplicated().sum())

print("Missing airline values:", flights["airline"].isna().sum())
print("UNKNOWN airline values:", (flights["airline"] == "UNKNOWN").sum())

print(
    "Non-uppercase source values:",
    (flights["source"] != flights["source"].str.upper()).sum()
)

print(
    "Non-uppercase destination values:",
    (flights["destination"] != flights["destination"].str.upper()).sum()
)

print(
    "Invalid duration records:",
    (flights["duration_hours"] < 0).sum()
)

Exact duplicate rows: 0
Missing airline values: 0
UNKNOWN airline values: 0
Non-uppercase source values: 0
Non-uppercase destination values: 0
Invalid duration records: 0


In [35]:
display(
    missing_flights[
        ["booking_id", "flight_id", "passenger_id", "status"]
    ]
)

,booking_id,flight_id,passenger_id,status


In [36]:
# Remove bookings linked to flights excluded from the analytical dataset
bookings = bookings[
    bookings["flight_id"].isin(flights["flight_id"])
].copy()

bookings_analytics = bookings_analytics[
    bookings_analytics["flight_id"].isin(flights["flight_id"])
].copy()

print("Bookings after removing orphan records:", len(bookings))

Bookings after removing orphan records: 999


In [37]:
# Remove payments that do not have a corresponding booking
payments = payments[
    payments["booking_id"].isin(bookings["booking_id"])
].copy()

print("Payments after removing orphan records:", len(payments))

Payments after removing orphan records: 999


In [38]:
# Validate referential integrity

missing_flights = bookings[
    ~bookings["flight_id"].isin(flights["flight_id"])
]

missing_passengers = bookings[
    ~bookings["passenger_id"].isin(passengers["passenger_id"])
]

missing_bookings = payments[
    ~payments["booking_id"].isin(bookings["booking_id"])
]

print("Bookings with missing flight:", len(missing_flights))
print("Bookings with missing passenger:", len(missing_passengers))
print("Payments with missing booking:", len(missing_bookings))

Bookings with missing flight: 0
Bookings with missing passenger: 0
Payments with missing booking: 0


In [39]:
# Final validation summary

print("=== FINAL DATA VALIDATION ===")

print("\nExact duplicates:")
print("Flights:", flights.duplicated().sum())
print("Bookings:", bookings.duplicated().sum())
print("Passengers:", passengers.duplicated().sum())
print("Payments:", payments.duplicated().sum())

print("\nFlight validation:")
print("Missing airline:", flights["airline"].isna().sum())
print("UNKNOWN airline:", (flights["airline"] == "UNKNOWN").sum())
print("Invalid duration:", (flights["duration_hours"] < 0).sum())

print("\nReferential integrity:")
print(
    "Missing flights:",
    (~bookings["flight_id"].isin(flights["flight_id"])).sum()
)
print(
    "Missing passengers:",
    (~bookings["passenger_id"].isin(passengers["passenger_id"])).sum()
)
print(
    "Missing bookings:",
    (~payments["booking_id"].isin(bookings["booking_id"])).sum()
)

=== FINAL DATA VALIDATION ===

Exact duplicates:
Flights: 0
Bookings: 0
Passengers: 0
Payments: 0

Flight validation:
Missing airline: 0
UNKNOWN airline: 0
Invalid duration: 0

Referential integrity:
Missing flights: 0
Missing passengers: 0
Missing bookings: 0


#### I have done all the preprocessing. I have verified that by final validation from duplicates missing value imputation and referential integrity now everything is looks fine

### 10. Final Level Analytical Dataset

In [40]:
# Flight-level analytical dataset
flights_analytics = flights.copy()

print("Flight analytical dataset:", flights_analytics.shape)
display(flights_analytics.head())

Flight analytical dataset: (1004, 8)


,flight_id,airline,source,destination,departure_time,arrival_time,duration,duration_hours
0,SJ010,SpiceJet,CCU,MAA,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,0 days 02:54:00,2.900000
1,AI155,Air India,BOM,CCU,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,0 days 01:48:00,1.800000
2,UK094,Vistara,BOM,CCU,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,0 days 01:45:00,1.750000
3,AI245,Air India,BOM,CCU,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704,0 days 02:36:00,2.600000
4,AI192,Air India,MAA,BOM,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,0 days 04:59:00,4.983333


In [41]:
flights_analytics["route"] = (
    flights_analytics["source"] + " → " +
    flights_analytics["destination"]
)

display(
    flights_analytics[
        ["flight_id", "source", "destination", "route"]
    ].head()
)

,flight_id,source,destination,route
0,SJ010,CCU,MAA,CCU → MAA
1,AI155,BOM,CCU,BOM → CCU
2,UK094,BOM,CCU,BOM → CCU
3,AI245,BOM,CCU,BOM → CCU
4,AI192,MAA,BOM,MAA → BOM


In [42]:
# Identify flight-level anomalies for analytical reporting

flights_analytics["anomaly_flag"] = "Normal"

# Flag unusually long flights for investigation
flights_analytics.loc[
    flights_analytics["duration_hours"] > 24,
    "anomaly_flag"
] = "Duration Anomaly"

display(
    flights_analytics[
        ["flight_id", "route", "duration_hours", "anomaly_flag"]
    ].head(10)
)

,flight_id,route,duration_hours,anomaly_flag
0,SJ010,CCU → MAA,2.900000,Normal
1,AI155,BOM → CCU,1.800000,Normal
2,UK094,BOM → CCU,1.750000,Normal
3,AI245,BOM → CCU,2.600000,Normal
4,AI192,MAA → BOM,4.983333,Normal
5,SJ158,DEL → HYD,2.316667,Normal
6,6F196,CCU → MAA,1.716667,Normal
7,AI080,BOM → HYD,1.533333,Normal
8,6F025,BLR → BOM,0.916667,Normal
9,6F251,DEL → BLR,0.683333,Normal


In [43]:
print(
    flights_analytics["anomaly_flag"].value_counts()
)

anomaly_flag
Normal    1004
Name: count, dtype: int64


In [44]:
flights_analytics["flight_date"] = (
    flights_analytics["departure_time"].dt.date
)

display(
    flights_analytics[
        ["flight_id", "flight_date", "airline", "route", "duration_hours"]
    ].head()
)

,flight_id,flight_date,airline,route,duration_hours
0,SJ010,2026-04-20,SpiceJet,CCU → MAA,2.900000
1,AI155,2026-04-20,Air India,BOM → CCU,1.800000
2,UK094,2026-04-20,Vistara,BOM → CCU,1.750000
3,AI245,2026-04-20,Air India,BOM → CCU,2.600000
4,AI192,2026-04-20,Air India,MAA → BOM,4.983333


In [45]:
flights_analytics.drop(
    columns=["anomaly_flag"],
    inplace=True
)

In [46]:
# Booking-level analytical dataset
bookings_analytics = bookings.copy()

print("Booking analytical dataset:", bookings_analytics.shape)

display(
    bookings_analytics.tail()
)

Booking analytical dataset: (999, 9)


,booking_id,passenger_id,flight_id,booking_date,status,passport_number,seat_number,emergency_contact_name,emergency_contact_phone
995,B1995,P1503,SJ005,2025-11-20 11:37:36.952,CANCELLED,J8784962,11C,Kiaan Sridhar,+91-8860378644
996,B1996,P1705,UK106,2025-08-08 11:37:36.952,CANCELLED,M1046250,27E,Michael Sha,+91-7709329827
997,B1997,P1122,6F223,2025-05-03 11:37:36.952,PENDING,P5685120,27B,Kashvi Sagar,+91-9548706636
998,B1998,P1647,6F011,2025-10-23 11:37:36.952,CONFIRMED,T2935097,14C,Urishilla Yogi,+91-8248695033
999,B1999,P1412,UK124,2025-08-08 11:37:36.952,INVALID,C2914650,12B,Tejas Kota,+91-7414292188


In [47]:
bookings_analytics["booking_date"] = pd.to_datetime(
    bookings_analytics["booking_date"],
    errors="coerce"
)

print("Booking date dtype:", bookings_analytics["booking_date"].dtype)

Booking date dtype: datetime64[ns]


In [48]:
print("Booking status distribution:")
display(bookings["status"].value_counts(dropna=False))

Booking status distribution:


status
CONFIRMED    319
CANCELLED    314
PENDING      291
NaN           45
INVALID       30
Name: count, dtype: int64

In [49]:
# Recreate booking analytical dataset from the cleaned bookings table
bookings_analytics = bookings.copy()

# Handle missing booking status
bookings_analytics["status"] = (
    bookings_analytics["status"].fillna("UNKNOWN")
)

# Remove emergency contact PII
bookings_analytics = bookings_analytics.drop(
    columns=["emergency_contact_name", "emergency_contact_phone"],
    errors="ignore"
)

print("Booking analytical dataset:", bookings_analytics.shape)
print("\nBooking status distribution:")
display(bookings_analytics["status"].value_counts(dropna=False))

Booking analytical dataset: (999, 7)

Booking status distribution:


status
CONFIRMED    319
CANCELLED    314
PENDING      291
UNKNOWN       45
INVALID       30
Name: count, dtype: int64

In [50]:
#AVERAGE FLIGHT DURATION
average_flight_duration = flights_analytics["duration_hours"].mean()

print(f"Average Flight Duration: {average_flight_duration:.2f} hours")

Average Flight Duration: 2.74 hours


In [51]:
#Route-wise Traffic
route_traffic = (
    flights_analytics
    .groupby("route")
    .size()
    .reset_index(name="flight_count")
    .sort_values("flight_count", ascending=False)
)

display(route_traffic.head(10))


,route,flight_count
6,BOM → CCU,90
12,CCU → DEL,72
25,MAA → BLR,65
0,BLR → BOM,60
24,HYD → MAA,57
18,DEL → HYD,54
23,HYD → DEL,42
7,BOM → DEL,39
11,CCU → BOM,33
15,DEL → BLR,29


In [52]:
# Airline-wise Flight Distribution
airline_distribution = (
    flights_analytics
    .groupby("airline")
    .size()
    .reset_index(name="flight_count")
    .sort_values("flight_count", ascending=False)
)

display(airline_distribution)

,airline,flight_count
1,IndiGo,273
0,Air India,255
2,SpiceJet,246
3,Vistara,230


In [53]:
Q1 = flights_analytics["duration_hours"].quantile(0.25)
Q3 = flights_analytics["duration_hours"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Q1: {Q1:.2f} hours")
print(f"Q3: {Q3:.2f} hours")
print(f"Lower bound: {lower_bound:.2f} hours")
print(f"Upper bound: {upper_bound:.2f} hours")

Q1: 1.65 hours
Q3: 3.89 hours
Lower bound: -1.71 hours
Upper bound: 7.24 hours


In [54]:
duration_anomalies = flights_analytics[
    flights_analytics["duration_hours"] > upper_bound
].copy()

print("Duration anomalies:", len(duration_anomalies))

display(
    duration_anomalies[
        ["flight_id", "airline", "source", "destination", "duration_hours"]
    ].sort_values("duration_hours", ascending=False)
)

Duration anomalies: 0


,flight_id,airline,source,destination,duration_hours


#### Total bookings and cancellation rate.

In [55]:
total_bookings = len(bookings_analytics)

cancelled_bookings = (
    bookings_analytics["status"] == "CANCELLED"
).sum()

cancellation_rate = (
    cancelled_bookings / total_bookings
) * 100

print("Total Bookings:", total_bookings)
print("Cancelled Bookings:", cancelled_bookings)
print(f"Cancellation Rate: {cancellation_rate:.2f}%")

Total Bookings: 999
Cancelled Bookings: 314
Cancellation Rate: 31.43%


#### Booking Status Distribution

In [56]:
booking_status_distribution = (
    bookings_analytics["status"]
    .value_counts()
    .reset_index()
)

booking_status_distribution.columns = [
    "status",
    "booking_count"
]

display(booking_status_distribution)

,status,booking_count
0,CONFIRMED,319
1,CANCELLED,314
2,PENDING,291
3,UNKNOWN,45
4,INVALID,30


In [57]:
print("Payment columns:")
print(payments.columns.tolist())

print("\nPayment dataset shape:")
print(payments.shape)

print("\nSample payment records:")
display(payments.head())

print("\nPayment data types:")
display(payments.dtypes)

Payment columns:
['payment_id', 'booking_id', 'amount', 'payment_method']

Payment dataset shape:
(999, 4)

Sample payment records:


,payment_id,booking_id,amount,payment_method
0,PAY1000,B1116,9883.49,NETBANKING
1,PAY1001,B1738,8457.96,NETBANKING
2,PAY1002,B1873,6495.37,UPI
3,PAY1003,B1914,5079.38,NETBANKING
4,PAY1004,B1967,12518.31,CARD



Payment data types:


payment_id        object
booking_id        object
amount            object
payment_method    object
dtype: object

In [58]:
print(payments.columns.tolist())
display(payments.head())

['payment_id', 'booking_id', 'amount', 'payment_method']


,payment_id,booking_id,amount,payment_method
0,PAY1000,B1116,9883.49,NETBANKING
1,PAY1001,B1738,8457.96,NETBANKING
2,PAY1002,B1873,6495.37,UPI
3,PAY1003,B1914,5079.38,NETBANKING
4,PAY1004,B1967,12518.31,CARD


In [59]:
payments["amount"] = pd.to_numeric(
    payments["amount"],
    errors="coerce"
)

print("Amount data type:", payments["amount"].dtype)
print("Missing amounts:", payments["amount"].isna().sum())
print("Zero amounts:", (payments["amount"] == 0).sum())
print("Negative amounts:", (payments["amount"] < 0).sum())

Amount data type: float64
Missing amounts: 78
Zero amounts: 0
Negative amounts: 0


In [60]:
invalid_amounts = payments[
    payments["amount"].isna()
]

print("Records with invalid/missing amount:", len(invalid_amounts))

display(
    invalid_amounts[
        ["payment_id", "booking_id", "amount", "payment_method"]
    ].head(20)
)

Records with invalid/missing amount: 78


,payment_id,booking_id,amount,payment_method
18,PAY1018,B1798,NaN,CARD
19,PAY1019,B1875,NaN,NETBANKING
24,PAY1024,B1523,NaN,CARD
26,PAY1026,B1120,NaN,CARD
33,PAY1033,B1012,NaN,CARD
60,PAY1060,B1572,NaN,NETBANKING
69,PAY1069,B1578,NaN,UPI
71,PAY1071,B1174,NaN,NETBANKING
77,PAY1077,B1501,NaN,UPI
173,PAY1173,B1722,NaN,CARD


In [61]:
payments_analytics = payments.copy()

payments_analytics["amount_status"] = (
    payments_analytics["amount"]
    .isna()
    .map({
        True: "Missing Amount",
        False: "Valid Amount"
    })
)

print(
    payments_analytics["amount_status"].value_counts()
)

display(
    payments_analytics[
        ["payment_id", "booking_id", "amount", "amount_status"]
    ].head(20)
)

amount_status
Valid Amount      921
Missing Amount     78
Name: count, dtype: int64


,payment_id,booking_id,amount,amount_status
0,PAY1000,B1116,9883.49,Valid Amount
1,PAY1001,B1738,8457.96,Valid Amount
2,PAY1002,B1873,6495.37,Valid Amount
3,PAY1003,B1914,5079.38,Valid Amount
4,PAY1004,B1967,12518.31,Valid Amount
5,PAY1005,B1998,7319.71,Valid Amount
6,PAY1006,B1544,8020.26,Valid Amount
7,PAY1007,B1043,8291.60,Valid Amount
8,PAY1008,B1508,7277.14,Valid Amount
9,PAY1009,B1987,4585.24,Valid Amount


In [62]:
total_payment_amount = payments_analytics["amount"].sum()

print(f"Total Payment Amount: ₹{total_payment_amount:,.2f}")

Total Payment Amount: ₹7,370,442.88


In [63]:
average_payment_amount = payments_analytics["amount"].mean()

print(f"Average Payment Amount: ₹{average_payment_amount:,.2f}")

Average Payment Amount: ₹8,002.65


In [64]:
#KPI summary table
kpi_summary = pd.DataFrame({
    "KPI": [
        "Total Flights",
        "Average Flight Duration (Hours)",
        "Total Bookings",
        "Cancelled Bookings",
        "Cancellation Rate (%)",
        "Total Payment Amount",
        "Average Payment Amount",
        "Statistical Duration Anomalies"
    ],
    "Value": [
        len(flights_analytics),
        round(average_flight_duration, 2),
        total_bookings,
        cancelled_bookings,
        round(cancellation_rate, 2),
        round(total_payment_amount, 2),
        round(average_payment_amount, 2),
        len(duration_anomalies)
    ]
})

display(kpi_summary)

,KPI,Value
0,Total Flights,1004.00
1,Average Flight Duration (Hours),2.74
2,Total Bookings,999.00
3,Cancelled Bookings,314.00
4,Cancellation Rate (%),31.43
5,Total Payment Amount,7370442.88
6,Average Payment Amount,8002.65
7,Statistical Duration Anomalies,0.00


In [65]:
import os

output_dir = "cleaned_data"
os.makedirs(output_dir, exist_ok=True)

flights_analytics.to_csv(
    f"{output_dir}/flights_clean.csv",
    index=False
)

bookings_analytics.to_csv(
    f"{output_dir}/bookings_clean.csv",
    index=False
)

passengers_analytics.to_csv(
    f"{output_dir}/passengers_clean.csv",
    index=False
)

payments_analytics.to_csv(
    f"{output_dir}/payments_clean.csv",
    index=False
)

kpi_summary.to_csv(
    f"{output_dir}/kpi_summary.csv",
    index=False
)

print("Cleaned datasets exported successfully.")
print("Files:", os.listdir(output_dir))

Cleaned datasets exported successfully.
Files: ['bookings_clean.csv', 'flights_clean.csv', 'kpi_summary.csv', 'passengers_clean.csv', 'payments_clean.csv']


In [66]:
print("=== FINAL DATA VALIDATION ===\n")

# 1. Exact duplicates
print("Exact duplicate rows:")
print("Flights:", flights_analytics.duplicated().sum())
print("Bookings:", bookings_analytics.duplicated().sum())
print("Passengers:", passengers_analytics.duplicated().sum())
print("Payments:", payments_analytics.duplicated().sum())

# 2. Missing key identifiers
print("\nMissing key identifiers:")
print("Flight ID:", flights_analytics["flight_id"].isna().sum())
print("Booking ID:", bookings_analytics["booking_id"].isna().sum())
print("Passenger ID:", passengers_analytics["passenger_id"].isna().sum())
print("Payment ID:", payments_analytics["payment_id"].isna().sum())

# 3. Flight duration
print("\nInvalid flight durations:")
print(
    (flights_analytics["duration_hours"] < 0).sum()
)

# 4. Referential integrity
missing_flights = bookings_analytics[
    ~bookings_analytics["flight_id"].isin(flights_analytics["flight_id"])
]

missing_passengers = bookings_analytics[
    ~bookings_analytics["passenger_id"].isin(
        passengers_analytics["passenger_id"]
    )
]

missing_bookings = payments_analytics[
    ~payments_analytics["booking_id"].isin(
        bookings_analytics["booking_id"]
    )
]

print("\nReferential integrity:")
print("Bookings with missing flights:", len(missing_flights))
print("Bookings with missing passengers:", len(missing_passengers))
print("Payments with missing bookings:", len(missing_bookings))

# 5. Payment amounts
print("\nPayment amount quality:")
print(
    "Missing amounts:",
    payments_analytics["amount"].isna().sum()
)
print(
    "Negative amounts:",
    (payments_analytics["amount"] < 0).sum()
)

# 6. PII protection checks
print("\nPII protection:")
print(
    "Passenger Aadhaar sample:",
    passengers_analytics["aadhaar_id"].iloc[0]
)

print(
    "Booking passport sample:",
    bookings_analytics["passport_number"].iloc[0]
)

print("\nValidation completed.")

=== FINAL DATA VALIDATION ===

Exact duplicate rows:
Flights: 0
Bookings: 0
Passengers: 0
Payments: 0

Missing key identifiers:
Flight ID: 0
Booking ID: 0
Passenger ID: 0
Payment ID: 0

Invalid flight durations:
0

Referential integrity:
Bookings with missing flights: 0
Bookings with missing passengers: 0
Payments with missing bookings: 0

Payment amount quality:
Missing amounts: 78
Negative amounts: 0

PII protection:
Passenger Aadhaar sample: 99466baa9b8fe69a6f31db34c87d50f0e9a488eda788f3fdce1c00e09a316285
Booking passport sample: P1945887

Validation completed.


In [67]:
# Protect passport numbers
bookings_analytics["passport_number"] = (
    bookings_analytics["passport_number"].apply(hash_value)
)

# Verify
print(
    "Passport sample:",
    bookings_analytics["passport_number"].iloc[0]
)

Passport sample: 59e7e7738c8711b2f49be2c355b25b9bd41b780392aa2151796e8b52aaa0a056


In [68]:
bookings_analytics.to_csv(
    "cleaned_data/bookings_clean.csv",
    index=False
)

print("Updated bookings_clean.csv with protected passport numbers.")

Updated bookings_clean.csv with protected passport numbers.
